# 📝 벡터 검색 과제 LV1(기초): 임베딩 적재와 두 가지 인덱스

> 이 단원의 새 기술을 하나씩 확인합니다. 교안에서는 약물 대사를 물었지만, 여기서는 **고혈압과 심혈관** 쪽으로 묻습니다. 문제는 네 갈래로 묶여 있습니다.
>
> - **1. 임베딩을 노드에 얹기**: `embed_texts` · `db.create.setNodeVectorProperty` · 저장된 벡터의 차원 재기
> - **2. 벡터 인덱스와 의미 검색**: `CREATE VECTOR INDEX`(차원·유사도) · `SEARCH ... SCORE AS` · 괄호 안 `LIMIT` 으로 정하는 top_k
> - **3. 전문 인덱스와 키워드 검색**: `CREATE FULLTEXT INDEX ... ON EACH` · `db.index.fulltext.queryNodes` · 하이픈이 든 기호와 따옴표 묶기
> - **4. 두 검색을 견주기**: 언제 의미로 찾고 언제 키워드로 찾나

## 풀이 방법
1. 맨 위 **준비 셀**과 **데이터 살펴보기**를 먼저 실행하세요(논문이 노드로 적재됩니다).
2. 문제는 **순서대로** 푸세요. 앞 문제에서 만든 임베딩·인덱스를 뒤 문제가 씁니다.
3. 각 문제의 **답안 셀**에 코드를 채우고 **자가채점 셀**로 확인하세요(✅ 통과!).

- **분량**: 코딩 10문 + 서술형 1문. 준비 셀부터 끝까지 60분 안팎 걸립니다.
- 데이터: `data/pmc_docs.jsonl`(각 행은 `pmcid·title·journal·year·license·text`).
- **이 과제는 실행 중인 Neo4j 가 필요합니다.** 반드시 **실습 전용** DB 에 연결하세요.
- 자가채점은 여러분이 담은 값을 **데이터베이스에서 다시 조회한 결과와 대조**합니다. 검색을 실제로 돌려야 통과합니다.

화이팅!

> **데이터 출처**: 이 단원의 데이터는 **공개된 원본을 값 그대로** 쓴 것입니다.
>
> | 데이터 | 원본 | 이용 조건 |
> |---|---|---|
> | 논문 69편 (`pmc_docs.jsonl`) | PubMed Central Open Access Subset. 각 행의 `pmcid` 가 원문 주소다 | **CC BY** |
> | 의료 지식그래프 (`hetionet_*.csv`) | Hetionet v1.0 (https://het.io) 에서 CC0 출처만 골라낸 부분 | **CC0** |
> | 이름 사전 (`name2id.json.gz`) | 위 Hetionet 이름 + RxNav(미국 국립의학도서관) 약물 동의어 | CC0 · NLM |
>
> 지식그래프는 **2016년에 정리된 자료**이고, 논문은 최근 것입니다. 그래서 이 둘을 이어 붙이면 그래프가 모르는 사실이 논문 쪽에 있습니다. 이 단원은 그 상태 그대로 검색합니다.
>
> 그리고 **논문이 보고했다**와 **효능이 입증됐다**는 다릅니다. 검색으로 찾은 문장을 답으로 옮길 때 이 구분을 놓치면, 근거가 있는 것처럼 보이는 틀린 답이 나옵니다.

아래 준비 셀을 먼저 실행하세요.

In [ ]:
# [제공 코드] OpenAI 키 준비 - 이 셀은 실행만 하세요.
# 14~16일차와 같은 방식입니다: .env 파일에 넣어 둔 OPENAI_API_KEY 를 읽어 옵니다.
import os

from dotenv import load_dotenv

load_dotenv(".env")       # 같은 폴더의 .env
load_dotenv("../.env")    # 정답 폴더에서 실행하는 경우

# 키를 먼저 확인합니다. 모델을 만든 뒤에 검사하면 인증 오류가 먼저 나서 이 안내가 묻힙니다.
if not os.getenv("OPENAI_API_KEY"):
    raise RuntimeError(
        "이 노트북은 실제 OpenAI 호출이 필요합니다 - OPENAI_API_KEY 를 찾지 못했습니다.\n"
        "  1) 일차 폴더에서  cp .env.example .env\n"
        "  2) .env 를 열어 본인 키를 채우세요\n"
        "  3) 커널을 재시작한 뒤 이 셀부터 다시 실행하세요")

print("OpenAI 키 확인 완료 - 이제 LangChain 으로 모델을 만들 수 있습니다.")

In [ ]:
# [제공 코드] 임베딩 준비: embed_texts(문장 리스트) 가 768차원 OpenAI 임베딩을 돌려줍니다(실행만 하세요).
# embed_texts(문서 리스트) 는 계산한 벡터를 data/emb_cache.pkl 에 저장해 두고 다시 씁니다(69편을 매번 다시 부르지 않으려고요).
# embed_query(질문 한 문장) 는 저장하지 않습니다. 질문은 매번 새로 만드는 것이 표준입니다.
import pickle
from pathlib import Path

from langchain_openai import OpenAIEmbeddings

EMBED_MODEL = "text-embedding-3-large"
EMBED_DIM = 768                    # 3072차원으로 나오는 모델을 768차원으로 잘라 받는다(아래 dimensions)
_EMB_FILE = Path("data/emb_cache.pkl") if Path("data").exists() else Path("../data/emb_cache.pkl")
_EMB_CACHE = pickle.loads(_EMB_FILE.read_bytes()) if _EMB_FILE.exists() else {}   # {문서: 벡터}
embedder = OpenAIEmbeddings(model=EMBED_MODEL, dimensions=EMBED_DIM)


def embed_texts(texts):
    """문서 리스트 -> 768차원 임베딩 리스트. 저장된 것은 그대로 쓰고, 없는 것만 임베딩해 저장한다."""
    new = [t for t in texts if t not in _EMB_CACHE]                  # 저장돼 있지 않은 문서만 고른다
    if new:
        _EMB_CACHE.update(zip(new, embedder.embed_documents(new)))   # 실제 호출은 이 줄뿐
        _EMB_FILE.write_bytes(pickle.dumps(_EMB_CACHE))              # 통째로 다시 저장
    return [_EMB_CACHE[t] for t in texts]


def embed_query(text):
    """질문 한 문장 -> 768차원 임베딩. 질문은 저장하지 않는다(매번 새로 만든다)."""
    return embedder.embed_query(text)


print("임베딩 모델:", EMBED_MODEL, f"({EMBED_DIM}차원) / 저장된 문서:", len(_EMB_CACHE), "건")

In [ ]:
# [제공 코드] Neo4j 연결: 변경 없이 그대로 실행하세요.
# - 로컬 실습 전용(포트 7689)으로 안전하게 연결됩니다.
# - 클라우드 Aura DB 접속은 가드에 의해 100% 원천 차단됩니다.
import os
from pathlib import Path
from dotenv import load_dotenv
from neo4j import GraphDatabase

# 1) 현재 폴더의 .env를 override=True로 강제 로드
env_candidates = [
    Path('.env'),
    Path('day35_벡터검색_GraphRAG/.env'),
    Path('내작업폴더/day35_벡터검색_GraphRAG/.env'),
    Path('../.env')
]
for p in env_candidates:
    if p.exists():
        load_dotenv(p, override=True)

# 로컬 실습 기본값 강제 (포트 7689)
NEO4J_URI = os.getenv('NEO4J_URI', 'bolt://localhost:7689')
NEO4J_USER = os.getenv('NEO4J_USER', 'neo4j')
NEO4J_PASSWORD = os.getenv('NEO4J_PASSWORD', 'test0011')

# 🚨 [안전 가드] 클라우드 DB 접속 원천 차단
if 'databases.neo4j.io' in str(NEO4J_URI):
    # 만약 환경변수에 클라우드가 남아있더라도 강제로 로컬 7689로 전환
    print('⚠️ 클라우드 Aura 주소가 감지되어 로컬 bolt://localhost:7689 로 자동 강제 전환합니다.')
    NEO4J_URI = 'bolt://localhost:7689'
    NEO4J_USER = 'neo4j'
    NEO4J_PASSWORD = 'test0011'

# 2) 드라이버 연결
driver = GraphDatabase.driver(NEO4J_URI, auth=(NEO4J_USER, NEO4J_PASSWORD))
driver.verify_connectivity()

# 3) Cypher 실행 헬퍼
def run_cypher(query, **params):
    with driver.session() as session:
        return [record.data() for record in session.run(query, **params)]

print('Neo4j 연결:', NEO4J_URI)


In [ ]:
# [제공 코드] 실습 전용 DB 초기화: 이 셀은 실행만 하세요.
# 몇 번이든 처음부터 다시 돌릴 수 있게 전부 내립니다. 반드시 "실습 전용" DB 여야 합니다.

# 1) GDS 투영: 노드를 지우기 전에 먼저 내린다. 투영은 원본 노드 id 를 기억하고 있어, 원본을 먼저 지우면 갈 곳을 잃는다
for _g in run_cypher("CALL gds.graph.list() YIELD graphName RETURN graphName"):
    run_cypher("CALL gds.graph.drop($name) YIELD graphName RETURN graphName", name=_g["graphName"])

# 2) 노드와 관계
run_cypher("MATCH (n) DETACH DELETE n")   # DETACH: 노드에 붙은 관계까지 함께 지운다

# 3) 벡터·전문 인덱스: 노드를 지워도 인덱스는 남는다. 차원이 다른 옛 인덱스가 남아 있으면 뒤에서 걸린다
for _ix in run_cypher("SHOW INDEXES YIELD name, type WHERE type IN ['VECTOR','FULLTEXT'] RETURN name"):
    run_cypher(f"DROP INDEX {_ix['name']} IF EXISTS")   # IF EXISTS: 이미 없어도 에러 없이 넘어간다

print("초기화 완료:", NEO4J_URI, "· 남은 노드:", run_cypher("MATCH (n) RETURN count(n) AS n")[0]["n"])

## 데이터 살펴보기
아래 셀은 **실행만** 하면 됩니다. 논문을 노드로 적재하고 한 편을 살펴봅니다(임베딩·인덱스는 아래 문제에서 직접 만듭니다).

> 이 과제는 검색 인덱스의 기초만 다루므로 **지식그래프는 올리지 않습니다.** 논문을 그래프와 잇는 일은 LV2 에서 합니다.

In [ ]:
# [제공 코드] 논문 코퍼스 적재: 이 셀은 실행만 하세요.
# PMC Open Access Subset 논문 69편(전 편 CC BY)을 Document 노드로 올립니다.
# text 는 초록에 본문 발췌를 이어 붙인 것입니다. 임베딩·인덱스는 아래 문제에서 직접 만듭니다.
import json
from pathlib import Path

DATA_DIR = Path("data") if Path("data").exists() else Path("../data")
papers = [json.loads(_line) for _line in
          (DATA_DIR / "pmc_docs.jsonl").read_text(encoding="utf-8").splitlines() if _line.strip()]
run_cypher("CREATE INDEX document_pmcid IF NOT EXISTS FOR (n:Document) ON (n.pmcid)")
run_cypher("UNWIND $rows AS row CREATE (n:Document) SET n += row", rows=papers)
print("적재한 논문:", run_cypher("MATCH (n:Document) RETURN count(n) AS c")[0]["c"], "편")


In [ ]:
# [제공 코드] 첫 논문 한 편을 살펴봅니다
sample = run_cypher("MATCH (n:Document) RETURN n.pmcid AS pmcid, n.title AS title, "
                    "n.year AS year, n.text AS text ORDER BY n.pmcid LIMIT 1")[0]
print(sample['pmcid'], '|', sample['year'], '|', sample['title'][:60])
print(sample['text'][:200].replace('\n', ' '), '...')

---
# 1. 임베딩을 노드에 얹기

논문 69편을 임베딩해 노드 속성으로 저장하고, 저장된 벡터에서 차원을 직접 재는 연습입니다(교안_01 1-3·2-1).

## 1-1. 논문 임베딩 적재하기
**배경**: 의미 검색을 하려면 먼저 각 논문을 **임베딩해서 노드에 저장**해야 합니다.

**요구사항**:
- 모든 `Document` 에 대해 **`title + ' ' + text`** 를 `embed_texts` 로 임베딩하세요.
- 각 노드에 `db.create.setNodeVectorProperty(n, 'emb', 벡터)` 로 저장하세요.
- 적재 후 `emb` 를 가진 `Document` 수를 **`loaded`** 에 담으세요.

**확인 기준**: `loaded` 는 정수입니다.

<details><summary>힌트</summary>

```text
접근방법:
- 모든 Document 의 pmcid·title·text 를 읽어와, 텍스트를 embed_texts 로 한 번에 임베딩하고, 노드별로 벡터를 저장한다.

세부구현:
1. MATCH 로 모든 Document 의 pmcid·title·text 를 한 번에 읽는다.
2. 제목과 본문을 공백으로 이은 문자열 리스트를 만들어 embed_texts 에 통째로 넘긴다(호출 한 번으로 끝난다).
3. 저장도 UNWIND 로 한 번에 보낸다. {pmcid, vec} 행을 만들어 넘기고 그 안에서 db.create.setNodeVectorProperty 를 부른다.
4. emb 가 NULL 이 아닌 Document 를 count 로 세어 loaded 에 담는다.
```

</details>

In [ ]:
# 여기에 코드를 작성하세요

In [ ]:
# [자가채점]
stored = run_cypher('MATCH (n:Document) WHERE n.emb IS NOT NULL RETURN count(n) AS c')[0]['c']
assert stored == 69, f'emb 가 저장된 Document 가 {stored}편입니다. 69편 모두에 벡터를 저장하세요'
assert loaded == stored, f'loaded({loaded})를 DB 에서 실제로 센 값({stored})으로 담으세요'
# 편수만 세면 제목만 임베딩해도 통과한다. 한 편을 꺼내 무엇을 임베딩했는지 대조한다
probe = run_cypher('MATCH (n:Document {pmcid:$p}) RETURN n.title AS t, n.text AS x, n.emb AS emb',
                   p='PMC13412467')[0]
want = embed_texts([probe['t'] + ' ' + probe['x']])[0]
assert max(abs(a - b) for a, b in zip(probe['emb'], want)) < 1e-6, \
    '저장된 벡터가 제목+본문을 임베딩한 것이 아닙니다. title 과 text 를 공백으로 이어 임베딩하세요'
print('✅ 통과!')

## 1-2. 임베딩 차원 확인하기
**배경**: 벡터 인덱스를 만들 때 **차원**을 정확히 맞춰야 합니다. 방금 저장한 임베딩이 몇 차원인지 확인합니다.

**요구사항**:
- `PMC13412467` 논문의 `emb` 속성 길이를 세어 **`dim`** 에 정수로 담으세요.

**확인 기준**: `dim` 은 정수입니다. 자가채점은 이 값을 저장된 벡터 길이와 대조하고, **2-1번에서 인덱스 차원으로 다시 씁니다.**

<details><summary>힌트</summary>

```text
접근방법:
- 그 논문 노드의 emb 를 읽어 파이썬 len 으로 길이를 잰다.

세부구현:
1. MATCH 로 그 pmcid 의 Document 를 찾아 emb 를 가져온다.
2. 돌려받은 값은 숫자 리스트다. 파이썬 len 으로 길이를 재 dim 에 담는다.
```

</details>

In [ ]:
# 여기에 코드를 작성하세요

In [ ]:
# [자가채점]
real = run_cypher('MATCH (n:Document {pmcid:$pmcid}) RETURN n.emb AS emb',
                  pmcid='PMC13412467')[0]['emb']
assert real is not None, '그 논문에 emb 가 없습니다. 1-1번을 먼저 푸세요'
assert dim == len(real), f'dim({dim})이 실제 벡터 길이({len(real)})와 다릅니다. 저장된 벡터를 읽어 재세요'
print('✅ 통과!')

---
# 2. 벡터 인덱스와 의미 검색

저장한 벡터를 인덱스에 걸고 `SEARCH` 절로 뜻이 가까운 논문을 찾는 연습입니다(교안_01 3절). 받을 편수는 괄호 안 `LIMIT` 이 정합니다.

## 2-1. 벡터 인덱스 만들기
**배경**: 의미 검색을 빠르게 하려면 **벡터 인덱스**가 필요합니다.

**요구사항**:
- `Document` 의 `emb` 속성에 대해 **이름 `doc_vec`**, **차원은 1-2번에서 잰 `dim`**, **유사도 `cosine`** 인 벡터 인덱스를 만드세요.
- 만든 뒤 `db.awaitIndexes()` 로 준비를 기다리세요.

**확인 기준**: 자가채점은 `SHOW INDEXES` 로 이름 `doc_vec` 의 종류(VECTOR)·대상(`Document` 의 `emb`)·차원(`dim`)·유사도 함수(`cosine`)를 확인합니다.

<details><summary>힌트</summary>

```text
접근방법:
- CREATE VECTOR INDEX 로 이름·레이블·속성·옵션(차원·유사도)을 지정하고, awaitIndexes 로 기다린다.

세부구현:
1. CREATE VECTOR INDEX 문에 인덱스 이름·레이블·속성을 적는다(두 번 실행해도 안전하도록 IF NOT EXISTS 를 붙인다).
2. OPTIONS 의 indexConfig 에 차원과 유사도 함수를 넣는다. 두 키는 점이 든 이름이라 백틱으로 감싼다(교안_01 3-1 의 문법 상자).
   차원 자리에는 숫자를 외워 적지 말고 1-2번의 dim 을 넣는다. 질의 문자열에 값을 박아야 하므로 f-string 을 쓴다(Cypher 의 중괄호는 `{{ }}` 로 두 번 적는다).
3. 마지막에 db.awaitIndexes 를 호출해 인덱스가 설 때까지 기다린다.
```

</details>

In [ ]:
# 여기에 코드를 작성하세요

In [ ]:
# [자가채점]
# 이름만 보면 차원과 유사도를 아무렇게나 줘도 통과한다. 인덱스 설정을 꺼내 지문의 세 조건을 다 본다
rows = run_cypher("SHOW INDEXES YIELD name, type, labelsOrTypes, properties, options "
                  "WHERE name = 'doc_vec' RETURN type, labelsOrTypes, properties, options")
assert rows, 'doc_vec 이 없습니다. 이름 철자와 awaitIndexes 실행을 확인하세요'
assert rows[0]['type'] == 'VECTOR', f"doc_vec 이 {rows[0]['type']} 인덱스입니다. 벡터 인덱스로 만드세요"
assert rows[0]['labelsOrTypes'] == ['Document'], \
    f"인덱스가 걸린 레이블이 {rows[0]['labelsOrTypes']} 입니다. Document 에 거세요"
assert rows[0]['properties'] == ['emb'], \
    f"인덱스가 걸린 속성이 {rows[0]['properties']} 입니다. emb 에 거세요"
config = rows[0]['options']['indexConfig']
assert config['vector.dimensions'] == dim, \
    f"인덱스 차원이 {config['vector.dimensions']} 인데 저장된 벡터는 {dim} 차원입니다"
assert config['vector.similarity_function'].lower() == 'cosine', \
    f"유사도가 {config['vector.similarity_function']} 입니다. cosine 으로 만드세요"
print('✅ 통과!')

## 2-2. 의미로 논문 찾기
**배경**: 누군가 **"고혈압 약은 어떻게 고르나요?"** 라고 물었습니다. 논문 제목에 그런 문장은 없지만, 뜻으로 가장 가까운 논문을 찾습니다.

**요구사항**:
- 질문 **`'고혈압 약은 어떻게 고르나요?'`** 를 임베딩해 `doc_vec` 인덱스로 **상위 1편**을 찾으세요.
- 그 논문의 `pmcid` 를 **`top_id`** 에 담으세요.
- `SEARCH` 절이 함께 돌려주는 **유사도 점수**를 **반올림하지 말고 그대로** **`top_score`** 에 담으세요.

**확인 기준**: `top_id` 는 문자열, `top_score` 는 0과 1 사이의 실수입니다(`(1+코사인)/2` 로 옮긴 값이라 0.67 언저리입니다. 교안_01 3-2). 소수 여섯째 자리까지 대조하므로 반올림한 값은 떨어집니다.

<details><summary>힌트</summary>

```text
접근방법:
- 질문을 embed_query 로 임베딩하고, MATCH 로 잡은 노드를 SEARCH 절에 넘겨 상위 문서를 받는다.

세부구현:
1. 질문 문장을 embed_query 에 넘겨 벡터를 받는다(문서는 embed_texts, 질문은 embed_query. 준비 셀 참고).
   (69편 임베딩은 data/emb_cache.pkl 에 있어 곧 끝난다.)
2. MATCH (n:Document) 뒤에 SEARCH 절을 붙인다. 괄호 안에 VECTOR INDEX 이름, FOR 뒤에 질문 벡터, LIMIT 에 받을 개수를 적는다.
3. 괄호 안에서 인덱스 이름·질문 벡터·받을 개수를 정하고, 점수 이름은 닫는 괄호 뒤에 붙인다(교안_01 3-2 의 문법 상자).
4. RETURN 에 pmcid 와 점수를 함께 담아, 첫 행에서 top_id 와 top_score 를 꺼낸다.
```

</details>

In [ ]:
# 여기에 코드를 작성하세요

In [ ]:
# [자가채점]
q = embed_query('고혈압 약은 어떻게 고르나요?')
truth = run_cypher('''MATCH (n:Document)
                        SEARCH n IN (VECTOR INDEX doc_vec FOR $q LIMIT 1) SCORE AS score
                      RETURN n.pmcid AS pmcid, score''', q=q)[0]
assert top_id == truth['pmcid'], (f"top_id({top_id})가 검색 결과({truth['pmcid']})와 다릅니다. "
                                  '검색 결과의 첫 행에서 그대로 꺼내세요')
assert top_id == 'PMC13461525', (f'1위가 {top_id} 로 나왔습니다. 이 코퍼스의 답은 PMC13461525 입니다. '
                              '질문 문장과 인덱스 이름을 확인하고, 그래도 같다면 1-1 에서 제목+본문이 아닌 '
                              '다른 것을 임베딩했을 수 있습니다')
assert 0 < top_score < 1, (f'top_score({top_score})가 0과 1 사이가 아닙니다. '
                           'SCORE AS 로 받은 값을 그대로 담으세요')
assert abs(top_score - truth['score']) < 1e-6, ('top_score 가 실제 검색 점수와 다릅니다. '
                                               '직접 적지 말고 SCORE AS 로 받은 값을 그대로 담으세요')
print('✅ 통과!')

## 2-3. 인덱스 없이 같은 답이 나오는지 확인하기
**배경**: 벡터 인덱스는 후보만 살피는 **근사 검색**이라 진짜 1위와 어긋날 수 있습니다(교안_01 3-1).

**요구사항**:
- 인덱스를 쓰지 말고 `vector.similarity.cosine(n.emb, $q)` 로 논문 69편을 **전부** 재세요.
- 점수가 가장 높은 논문의 `pmcid` 를 **`brute_id`** 에 담으세요.
- 그 점수를 **반올림하지 말고 그대로** **`brute_score`** 에 담고, 두 값을 출력하세요.

**확인 기준**: `brute_id` 는 2-2번의 `top_id` 와 같고, `brute_score` 도 `top_score` 와 소수 여섯째 자리까지 같습니다.

<details><summary>힌트</summary>

```text
접근방법:
- SEARCH 절을 쓰지 않고 MATCH 로 모든 Document 를 훑어 함수로 점수를 재고 정렬한다.

세부구현:
1. 2-2번과 같은 질문을 embed_query 로 임베딩한다.
2. MATCH (n:Document) 뒤 RETURN 에서 vector.similarity.cosine(n.emb, $q) 를 점수로 계산한다.
3. ORDER BY 점수 DESC LIMIT 1 로 한 편만 받아 pmcid 와 점수를 꺼낸다.
```

</details>

In [ ]:
# 여기에 코드를 작성하세요

In [ ]:
# [자가채점]
assert brute_id == top_id, (f'brute_id({brute_id})가 2-2번의 top_id({top_id})와 다릅니다. '
                            '69편을 전부 재고 점수 내림차순 1위를 담으세요')
assert abs(brute_score - top_score) < 1e-6, \
    (f'brute_score({brute_score})가 2-2번의 top_score({top_score})와 다릅니다. '
     'vector.similarity.cosine 은 SEARCH 절과 같은 눈금이라 값이 같아야 합니다')
print('✅ 통과!')

## 2-4. 상위 여러 편 찾기(top_k)
**배경**: 이번엔 **"심혈관 위험을 줄이는 방법"** 라는 질문으로 가까운 논문 **상위 3편**을 받습니다.

**요구사항**:
- 질문 **`'심혈관 위험을 줄이는 방법'`** 로 상위 **3편**을 찾아 `pmcid` 를 **점수 내림차순 그대로** **`top3`** 에 리스트로 담으세요.

**확인 기준**: `top3` 은 길이 3의 문자열 리스트입니다. 자가채점은 같은 검색을 다시 돌려 **3위까지의 순서 전체**를 봅니다.

<details><summary>힌트</summary>

```text
접근방법:
- 2-2번과 같되 받을 개수를 3으로 주고, 결과 pmcid 를 점수 내림차순 순서대로 리스트로 모은다.

세부구현:
1. 질문을 embed_query 로 임베딩한다.
2. 2-2번과 같은 SEARCH 절을 쓰되, 괄호 안 LIMIT 을 3 으로 준다.
3. score 내림차순으로 정렬해 pmcid 를 순서대로 리스트에 담는다.
```

</details>

In [ ]:
# 여기에 코드를 작성하세요

In [ ]:
# [자가채점]
q = embed_query('심혈관 위험을 줄이는 방법')
truth = [row['pmcid'] for row in run_cypher('''MATCH (n:Document)
                                                SEARCH n IN (VECTOR INDEX doc_vec FOR $q LIMIT 3)
                                                SCORE AS score
                                              RETURN n.pmcid AS pmcid ORDER BY score DESC''', q=q)]
assert len(top3) == 3, f'상위 3편을 담아야 합니다(현재 {len(top3)}편). SEARCH 안의 LIMIT 을 3 으로 주세요'
assert top3[0] == 'PMC13461525', f'1위가 다릅니다: {top3[0]}. 질문 문장을 확인하세요'
assert top3 == truth, '2·3위 순서가 실제 검색 결과와 다릅니다. score 내림차순 그대로 담으세요'
print('✅ 통과!')

## 2-5. 조건을 `SEARCH` 밖에 두느냐 안에 두느냐
**배경**: **`Frontiers in Medicine`** 에 실린 논문만 보고 싶습니다.

**요구사항**:
- 2-2번의 질문으로 `doc_vec` 상위 **5편**을 받은 뒤 `journal` 이 **`'Frontiers in Medicine'`** 인 것만 남겨 그 편수를 **`post_count`** 에 담으세요(조건을 `SEARCH` **밖**에).
- 거를 속성을 함께 저장한 벡터 인덱스 **`doc_vec_journal`** 을 만드세요(`doc_vec` 과 차원·유사도는 같게 하고, `journal` 을 거를 수 있게 등록합니다).
- 같은 질문·같은 조건을 이번엔 `SEARCH` **안**에 넣어 상위 5편을 받아 편수를 **`pre_count`** 에 담고, 두 값을 함께 출력하세요.

**확인 기준**: 두 값은 정수이고 **서로 다릅니다.** 사전 필터 쪽이 큽니다. 자가채점은 두 값을 각각 실제 검색 결과와 대조하고, `doc_vec_journal` 에 거를 속성이 등록됐는지도 확인합니다.

<details><summary>힌트</summary>

```text
접근방법:
- 같은 조건을 SEARCH 밖과 안에 각각 두고 편수를 견준다.

세부구현:
1. 사후 필터는 2-2번 질의 뒤에 WHERE 를 한 줄 붙이면 된다.
2. 사전 필터를 쓰려면 인덱스를 만들 때 거를 속성을 함께 저장해야 한다. 2-1번 CREATE 문의 ON n.emb 뒤에 그 자리가 있다(교안_01 3-2).
3. 새 인덱스도 db.awaitIndexes 로 기다린다.
4. 사전 필터는 조건을 SEARCH 괄호 안, LIMIT 앞에 둔다.
```

</details>

In [ ]:
# 여기에 코드를 작성하세요

In [ ]:
# [자가채점]
# 인덱스에 거를 속성이 등록됐는지부터 본다. 등록 안 하고 SEARCH 안에 조건을 넣으면 그 자리에서 에러가 난다
ix = run_cypher("SHOW INDEXES YIELD name, type, properties WHERE name = 'doc_vec_journal' "
                "RETURN type, properties")
assert ix, 'doc_vec_journal 이 없습니다. 이름 철자와 awaitIndexes 실행을 확인하세요'
assert 'journal' in ix[0]['properties'], \
    (f"doc_vec_journal 이 든 속성이 {ix[0]['properties']} 입니다. "
     'ON n.emb 뒤에 거를 속성을 함께 저장하세요')
# 두 값도 리터럴을 믿지 않고 그 자리에서 다시 검색해 대조한다
_qv, _j = embed_query('고혈압 약은 어떻게 고르나요?'), 'Frontiers in Medicine'
_post = run_cypher('''MATCH (n:Document)
                        SEARCH n IN (VECTOR INDEX doc_vec FOR $q LIMIT 5) SCORE AS score
                      WHERE n.journal = $j RETURN n.pmcid AS p''', q=_qv, j=_j)
_pre = run_cypher('''MATCH (n:Document)
                       SEARCH n IN (VECTOR INDEX doc_vec_journal FOR $q WHERE n.journal = $j LIMIT 5)
                       SCORE AS score
                     RETURN n.pmcid AS p''', q=_qv, j=_j)
assert post_count == len(_post), (f'post_count 가 {post_count} 인데 검색해 보면 {len(_post)} 편입니다. '
                                  '조건을 SEARCH 괄호 밖에 두고 상위 5편을 먼저 받았는지 확인하세요')
assert pre_count == len(_pre), (f'pre_count 가 {pre_count} 인데 검색해 보면 {len(_pre)} 편입니다. '
                                '조건을 SEARCH 괄호 안, LIMIT 앞에 두었는지 확인하세요')
print('✅ 통과!')

---
# 3. 전문 인덱스와 키워드 검색

같은 논문 더미를 이번에는 **글자로** 찾는 연습입니다(교안_01 4절). 뜻이 아니라 키워드가 맞아야 걸리고, 그 키워드를 인덱스가 어떻게 쪼갰느냐가 결과를 바꿉니다.

## 3-1. 전문(키워드) 인덱스 만들기
**배경**: 이번엔 **글자가 든 논문**을 정확히 찾는 전문 인덱스를 만듭니다.

**요구사항**:
- `Document` 의 `title` 과 `text` 를 대상으로 **이름 `doc_ft`** 인 전문 인덱스를 만드세요.
- 만든 뒤 `db.awaitIndexes()` 로 준비를 기다리세요.

**확인 기준**: 자가채점은 `SHOW INDEXES` 로 이름 `doc_ft` 의 종류(FULLTEXT)와 대상 속성(`title`·`text`)을 확인합니다.

<details><summary>힌트</summary>

```text
접근방법:
- CREATE FULLTEXT INDEX 로 이름·레이블·대상 속성 목록을 지정하고 awaitIndexes 로 기다린다.

세부구현:
1. CREATE FULLTEXT INDEX 문에 인덱스 이름·레이블을 적고, ON EACH 뒤 대괄호에 색인할 두 속성을 나열한다.
2. 마지막에 db.awaitIndexes 를 호출한다.
```

</details>

In [ ]:
# 여기에 코드를 작성하세요

In [ ]:
# [자가채점]
# 이름만 보면 제목만 색인해도 통과하고, 그 잘못이 3-2번에 가서야 드러난다. 대상 속성까지 본다
rows = run_cypher("SHOW INDEXES YIELD name, type, properties WHERE name = 'doc_ft' "
                  "RETURN type, properties")
assert rows, 'doc_ft 가 없습니다. 이름 철자와 awaitIndexes 실행을 확인하세요'
assert rows[0]['type'] == 'FULLTEXT', \
    f"doc_ft 가 {rows[0]['type']} 인덱스입니다. 전문 인덱스로 만드세요"
assert sorted(rows[0]['properties']) == ['text', 'title'], \
    f"색인한 속성이 {rows[0]['properties']} 입니다. ON EACH 에 title 과 text 를 둘 다 넣으세요"
print('✅ 통과!')

## 3-2. 키워드로 논문 찾기
**배경**: **`hypertension`** 이라는 키워드가 든 논문을 전문 인덱스로 찾습니다. 몇 편이 나올지는 검색해 봐야 압니다.

**요구사항**:
- `doc_ft` 인덱스로 **`'hypertension'`** 을 검색해, 나온 논문들의 `pmcid` 를 **점수 내림차순 그대로** **`word_ids`** 에 리스트로 담으세요.
- 검색어를 **`'hypertension AND genotype'`** 으로 바꿔 나온 편수를 **`and_count`** 에 담으세요.
- **`'hypertension OR genotype'`** 의 편수를 **`or_count`** 에 담으세요.
- 제목에서만 찾는 **`'title:hypertension'`** 의 편수를 **`title_count`** 에 담고, 네 값을 함께 출력하세요.

**확인 기준**: `word_ids` 는 문자열 리스트입니다(한 편이 아닙니다). 자가채점은 같은 검색을 다시 돌려 **건수와 1위, 그리고 담긴 논문 전체**를 대조합니다. 꼬리 쪽에 **점수가 똑같은 두 편**이 있어 그 둘의 앞뒤는 점수가 정해 주지 않으므로, 그 순서는 채점하지 않습니다.

뒤의 세 값은 정수이고, `and_count` 는 `or_count` 보다 작거나 같습니다.

<details><summary>힌트</summary>

```text
접근방법:
- db.index.fulltext.queryNodes 에 인덱스명과 검색어를 넘겨 논문을 받고, pmcid 를 리스트로 모은다.

세부구현:
1. 그 프로시저를 CALL 한다. 인자는 인덱스 이름과 검색어 두 개이고, YIELD 로 node 와 score 를 받는다.
2. score 내림차순으로 정렬해 노드 pmcid 를 RETURN 하고, 결과에서 pmcid 만 뽑아 리스트로 만든다.
3. 뒤의 세 값은 같은 프로시저에 검색어만 바꿔 넣고 결과 길이를 세면 된다. 검색어 문법은 교안_01 4-1 에 있다.
```

</details>

In [ ]:
# 여기에 코드를 작성하세요

In [ ]:
# [자가채점]
truth = [row['pmcid'] for row in run_cypher('''CALL db.index.fulltext.queryNodes('doc_ft', $term)
                                              YIELD node, score
                                              RETURN node.pmcid AS pmcid ORDER BY score DESC''',
                                           term='hypertension')]
assert len(truth) == 8, f'준비 상태가 다릅니다. 3-1번 인덱스를 다시 확인하세요(현재 {len(truth)}편)'
assert len(word_ids) == len(truth), (f'검색 결과는 {len(truth)}편인데 {len(word_ids)}편을 담았습니다. '
                                     '전문 검색을 실제로 돌려 나온 것을 모두 담으세요')
# 점수가 같은 두 편이 있어 그 둘의 앞뒤는 검색어만으로 정해지지 않는다. 1위와 집합으로만 본다
assert word_ids[0] == truth[0], f'1위가 다릅니다: {word_ids[0]}. score 내림차순으로 담으세요'
assert set(word_ids) == set(truth), ('건수는 맞지만 담긴 논문이 다릅니다. '
                                          'doc_ft 로 그 키워드를 검색해 나온 pmcid 를 그대로 담으세요')
# 세 값도 리터럴을 믿지 않고 그 자리에서 다시 검색해 대조한다
def _n(term):
    return len(run_cypher('CALL db.index.fulltext.queryNodes(\'doc_ft\', $t) YIELD node '
                          'RETURN node.pmcid AS p', t=term))
for _name, _got, _term in [('and_count', and_count, 'hypertension AND genotype'),
                           ('or_count', or_count, 'hypertension OR genotype'),
                           ('title_count', title_count, 'title:hypertension')]:
    assert _got == _n(_term), (f'{_name} 이 {_got} 인데 검색해 보면 {_n(_term)} 편입니다. '
                               f'검색어 {_term!r} 를 그대로 넣고 결과 길이를 세세요')
assert or_count == 15, (f'or_count 가 {or_count} 입니다. OR 은 둘 중 하나라도 든 논문이라 '
                               '8편과 7편의 합이 됩니다')
print('✅ 통과!')

## 3-3. 하이픈이 든 기호를 키워드 검색하기
**배경**: 전문 인덱스는 문장을 단어로 쪼개 저장하는데, 그 규칙이 하이픈을 단어 경계로 봅니다(교안_01 4-3). 사이토카인 기호 **`IL-6`** 으로 확인합니다.

**요구사항**:
- `doc_ft` 로 **`'IL-6'`** 을 그대로 검색해 나온 편수를 **`loose_count`** 에 담으세요.
- 같은 기호를 **큰따옴표로 묶어**(`'"IL-6"'`) 검색해 나온 편수를 **`exact_count`** 에 담으세요.
- 두 값을 출력하세요.

**확인 기준**: 두 값이 **다릅니다.** 묶지 않은 쪽이 더 큽니다.

<details><summary>힌트</summary>

```text
접근방법:
- 3-2번과 같은 프로시저를 검색어만 바꿔 두 번 부르고, 각각 나온 행 수를 센다.

세부구현:
1. 검색어를 받아 결과 편수를 돌려주는 작은 함수를 하나 만든다.
2. 기호를 그대로 넣어 한 번, 앞뒤에 큰따옴표를 붙여 한 번 부른다.
3. 파이썬 문자열 안에 큰따옴표를 넣으려면 작은따옴표로 감싸면 편하다.
```

</details>

In [ ]:
# 여기에 코드를 작성하세요

In [ ]:
# [자가채점]
def _count(term):
    return len(run_cypher('''CALL db.index.fulltext.queryNodes('doc_ft', $term) YIELD node
                            RETURN node.pmcid AS pmcid''', term=term))


assert loose_count == _count('IL-6'), '묶지 않은 검색 결과를 실제로 세어 담으세요'
assert exact_count == _count('"IL-6"'), \
    '따옴표로 묶은 검색 결과를 실제로 세어 담으세요'
assert loose_count > exact_count, ('묶지 않은 쪽이 더 많아야 합니다. '
                                   '두 검색어를 바꿔 넣지 않았는지 확인하세요')
print('✅ 통과!')

---
# 4. 두 검색을 견주기

앞의 두 갈래에서 같은 논문 더미를 두 방법으로 검색해 봤습니다. 이제 언제 어느 쪽을 쓸지 말로 정리합니다(교안_01 4-2).

## 4-1. 서술형: 어느 검색을 쓸 것인가
**배경**: 같은 논문 더미를 벡터 인덱스(의미)와 전문 인덱스(키워드)로 검색해 봤습니다. 2-4번의 상위 3편과 3-2번의 8편을 나란히 놓으면 겹치는 것이 하나도 없습니다.

**요구사항**: 아래 두 상황에서 **어느 검색이 더 나은지**와 **그 이유**를 각각 한두 문장으로 쓰세요.
1. 의사가 **"혈압약을 고를 때 유전자 검사가 도움이 되나"** 처럼 풀어서 물어볼 때.
2. 연구자가 **`IL-6` 이 언급된 논문만 정확히** 모으려 할 때.

아래 markdown 셀에 답을 쓰세요. 정답 노트북의 모범 서술과 비교해 보세요.

*(1번 상황과 2번 상황 각각에 대해 '어느 검색' + '왜' 를 한두 문장씩 쓰세요)*